# Module 8: Benchmark and Evaluate

Every earlier module moved one number. Module 4 found the throughput knee, Module 5 pushed it, Module 6 traded size for accuracy, Module 7 split the card. A deployment decision is never one number. It is speed, cost, and accuracy together, and you have to defend the tradeoff. This module runs a structured sweep against your [vLLM](https://docs.vllm.ai) server, turns measured throughput into dollars per million tokens, scores quality on a small evaluation set, and writes down a recommended operating point you could put in front of your team. The running example is the inference server underneath your agents, served on an [Akamai Cloud GPU](https://www.linode.com/products/gpu/).

## Learning objectives
- Run a concurrency sweep and keep throughput and latency at every level
- Plot throughput against latency and draw your SLO as a line on it
- Read the operating point off the curve: the rightmost point under the line
- Turn measured throughput plus instance price into cost per million tokens
- Score quality on a small evaluation set so accuracy sits beside speed and cost
- Write down one defensible operating point with all three numbers
- Tell a benchmark number apart from an SLO

## Prerequisites
- Finished Module 4 and ideally Module 5, with a tuned server to benchmark
- A live vLLM endpoint in `VLLM_HOST`, resolved by `common/config.py`
- The `vllm` CLI for `vllm bench serve`, or none: the sweep falls back to a pure-Python load generator automatically
- About 20 minutes

References: [vLLM benchmarks](https://docs.vllm.ai/en/latest/contributing/benchmarks.html) &middot; [GuideLLM](https://github.com/vllm-project/guidellm) &middot; [GuideLLM walkthrough](https://developers.redhat.com/articles/2025/06/20/guidellm-evaluate-llm-deployments-real-world-inference) &middot; [Single-GPU vLLM reference numbers](https://www.databasemart.com/blog/vllm-gpu-benchmark-a6000) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Benchmark versus SLO design basics

A benchmark and a service level objective are not the same thing, and confusing them is the failure this module fixes.

- **A benchmark number** is what the server can do at one chosen load. "1,200 tokens per second at concurrency 64" is a benchmark. It is a fact about the machine.
- **An SLO** is the promise you make to users, usually a latency ceiling at a percentile: "p95 time to first token under 500 ms." It is a fact about what you will tolerate.

A single benchmark number, picked to look good, can hide a latency your users feel. The sweep gives you the whole curve; the SLO picks the point on it you are allowed to run at. The operating point is the highest throughput you can buy without breaking the promise. Push past it and you are paying for tokens in latency your users notice.

From there, two more numbers fall out of that one point. Throughput plus the instance price gives cost per million tokens. The same configuration, run against a small evaluation set, gives accuracy. Speed, cost, and accuracy at one point: that is the decision.

![A throughput versus latency curve with a horizontal p95 TTFT SLO line, the operating point at the rightmost point under the line, and two derived numbers, dollars per million tokens and eval accuracy, forming a three-number decision](images/08_benchmark_and_evaluate_architecture.png)

## 1. Setup

This module reads its connection details from `common/config.py` and uses the shared `metrics` and `load` helpers. Install the one plotting dependency this notebook adds. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q matplotlib

## 2. Configure endpoint and load tool

`get_settings()` reads `VLLM_HOST` and `MODEL_NAME` from your environment. The sweep runs against this endpoint, so benchmark the exact configuration you intend to ship. The hosted image ships the `vllm` CLI; if it is absent, the code below detects that and falls back to the pure-Python load generator in `common/load.py`, which returns the same fields.

In [ ]:
# Resolve settings and pick the load tool: the vllm CLI if present, else the
# pure-Python fallback in common/load.py. Either path returns the same fields.
import os, sys, json, shutil, subprocess, tempfile
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings, build_client
from common import load

settings = get_settings()
server_root = settings.vllm_host.rstrip("/")
if server_root.endswith("/v1"):
    server_root = server_root[:-3]
# The hosted image ships the `vllm` CLI; without it the sweep falls back to the
# pure-Python load generator in common/load.py automatically.
HAVE_VLLM_CLI = shutil.which("vllm") is not None
print("server:", server_root, "| model:", settings.model_name,
      "| load tool:", "vllm bench serve" if HAVE_VLLM_CLI else "pure-Python fallback")

**What you should see:** your server root, model, and which load tool will run. The sweep hits this endpoint, so this is the configuration you are deciding on.

## 3. Run the sweep

Same load tool as Module 4, run across a range of concurrency levels to build the whole curve. The difference is that you keep every row this time, because the analysis afterward needs throughput and latency at each point. `run_level` uses `vllm bench serve` when the CLI is present and the pure-Python generator otherwise, and returns the same dictionary either way.

In [ ]:
# Requires a live vLLM endpoint. Uses the vllm CLI when present, otherwise the
# pure-Python load generator in common/load.py (same fields either way).
# Run one concurrency level and return the metrics needed for the analysis.
def run_level(concurrency, num_prompts=None, input_len=256, output_len=256):
    if not HAVE_VLLM_CLI:
        return load.run_level(
            build_client(settings), settings.model_name, settings.metrics_url,
            concurrency, num_prompts=num_prompts, output_len=output_len,
        )

    num_prompts = num_prompts or max(concurrency * 4, 16)
    out_path = os.path.join(tempfile.gettempdir(), f"eval_c{concurrency}.json")
    cmd = [
        "vllm", "bench", "serve", "--backend", "openai-chat",
        "--base-url", server_root, "--endpoint", "/v1/chat/completions",
        "--model", settings.model_name, "--dataset-name", "random",
        "--random-input-len", str(input_len), "--random-output-len", str(output_len),
        "--num-prompts", str(num_prompts), "--max-concurrency", str(concurrency),
        "--save-result", "--result-filename", out_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stderr[-400:]); raise RuntimeError("bench failed")
    with open(out_path) as f:
        data = json.load(f)
    return {
        "concurrency": concurrency,
        "output_throughput": data.get("output_throughput"),     # tokens/s
        "p95_ttft_ms": data.get("p95_ttft_ms") or data.get("p99_ttft_ms"),
        "mean_ttft_ms": data.get("mean_ttft_ms"),
        "p95_tpot_ms": data.get("p95_tpot_ms") or data.get("mean_tpot_ms"),
    }

levels = [1, 4, 8, 16, 32, 64]
rows = []
for c in levels:
    r = run_level(c); rows.append(r)
    print(f"c={c:>3}  {r['output_throughput']:>7.0f} tok/s  "
          f"p95 TTFT={r['p95_ttft_ms']:>7.0f} ms")

**What you should see:** one row per concurrency level, with output throughput and p95 TTFT. Throughput climbs then flattens; p95 TTFT stays low then rises. Those two columns drive every decision below.

## 4. Plot throughput against latency

The most useful view puts throughput on one axis and your latency metric on the other, with concurrency as the path along the curve. Your SLO is a horizontal line. The operating point is the rightmost point still under that line: the most throughput you can serve while keeping the promise.

In [ ]:
# Plot the throughput vs p95 TTFT tradeoff, with the SLO line.
import matplotlib.pyplot as plt

SLO_TTFT_MS = 500   # your latency objective; edit to your target

thru = [r["output_throughput"] for r in rows]
ttft = [r["p95_ttft_ms"] for r in rows]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thru, ttft, marker="o")
for r in rows:
    ax.annotate(f"c={r['concurrency']}", (r["output_throughput"], r["p95_ttft_ms"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.axhline(SLO_TTFT_MS, color="tab:red", linestyle="--", label=f"SLO p95 TTFT {SLO_TTFT_MS} ms")
ax.set_xlabel("output throughput (tokens/s)")
ax.set_ylabel("p95 TTFT (ms)")
ax.set_title("Throughput vs latency, labeled by concurrency")
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

**What you should see:** a curve sweeping up and to the right, each point labeled with its concurrency, and a red SLO line across it. Points below the line meet your objective. The one furthest right under the line is the most throughput you can serve without breaking the promise. That point is your candidate.

## 5. Turn throughput into cost per million tokens

Throughput plus the price of the instance gives cost per token, which is how you compare against a hosted API on equal terms. The math is plain: a GPU instance costs a fixed amount per hour and produces a measured number of tokens per second. Divide. This is the number Module 1 promised the sweep would give you.

In [ ]:
# Compute cost per million output tokens at each operating point.
# Two prices: the ideal at full load, and the honest one at real utilization.
INSTANCE_HOURLY = 1.50   # dollars per hour for the GPU instance
UTILIZATION     = 0.70   # the share of hours the card actually runs this load

print(f"{'conc':>5} {'tok/s':>8} {'$/1M peak':>10} {'$/1M real':>10} {'p95 TTFT ms':>12}")
for r in rows:
    tok_s = r["output_throughput"]
    cost_peak = (INSTANCE_HOURLY / (tok_s * 3600)) * 1e6 if tok_s else float("inf")
    cost_real = cost_peak / UTILIZATION if tok_s else float("inf")
    print(f"{r['concurrency']:>5} {tok_s:>8.0f} {cost_peak:>10.3f} {cost_real:>10.3f} {r['p95_ttft_ms']:>12.0f}")

**What you should see:** a table where cost per million tokens falls as concurrency rises, because the fixed hourly cost spreads over more tokens. This is the core economic argument for owning inference: at high utilization your cost per token drops well below per-token hosted pricing. Compare the cheapest row here against the hosted estimate you made in Module 1. The real column divides by utilization, because a card that sits idle part of the day still bills for the whole day. A GPU at 10 percent utilization costs roughly 10x its busy-hour rate per token. That is the honest version of the Module 1 crossover: owning wins only when the card stays busy.

## 6. Score quality on a small eval set

Speed and cost are worthless if the answers are wrong. A small evaluation set gives you a quality number to set against them: a handful of questions with checkable answers. For your own use, swap these for prompts that look like your real traffic.

In [ ]:
# Requires a live vLLM endpoint.
# Score the served model on a small fixed evaluation set.
client = build_client(settings)

EVAL = [
    ("What is the capital of Japan?", "tokyo"),
    ("What is 15 times 6?", "90"),
    ("What gas do plants absorb during photosynthesis?", "carbon dioxide"),
    ("How many continents are there?", "7"),
    ("What is the largest planet in our solar system?", "jupiter"),
    ("What is the freezing point of water in Celsius?", "0"),
    ("Who wrote the play Romeo and Juliet?", "shakespeare"),
    ("What is the square root of 81?", "9"),
    ("What is the chemical symbol for gold?", "au"),
    ("What is the opposite of 'increase'?", "decrease"),
]

correct = 0
for q, expected in EVAL:
    resp = client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": q}],
        max_tokens=40, temperature=0.0,
    )
    ans = resp.choices[0].message.content.strip().lower()
    hit = expected.replace(" ", "") in ans.replace(" ", "")
    correct += hit

accuracy = correct / len(EVAL)
print(f"eval accuracy: {correct}/{len(EVAL)} = {accuracy*100:.0f}%")

**What you should see:** an accuracy score on the small set, around 90% for a 4B instruct model on questions this easy. Run this against each configuration you compare, full precision versus quantized, one model versus another, so quality lands in the same table as speed and cost. A real evaluation uses a larger, task-specific set, but the method is identical.

## 7. Write down the operating point

The deliverable is one recommendation: the configuration and concurrency you would run, with the three numbers that justify it. Pick the highest-throughput level that meets your SLO, read its cost, and pair it with the quality score. This is the artifact you defend in a review.

In [ ]:
# Pick the best operating point under the SLO and print the recommendation.
under_slo = [r for r in rows if r["p95_ttft_ms"] and r["p95_ttft_ms"] <= SLO_TTFT_MS]
if under_slo:
    best = max(under_slo, key=lambda r: r["output_throughput"])
    cost_per_m = (INSTANCE_HOURLY / (best["output_throughput"] * 3600)) * 1e6
    print("Recommended operating point")
    print("-" * 40)
    print(f"  model        : {settings.model_name}")
    print(f"  concurrency  : {best['concurrency']}")
    print(f"  throughput   : {best['output_throughput']:.0f} tokens/s")
    print(f"  p95 TTFT     : {best['p95_ttft_ms']:.0f} ms (SLO {SLO_TTFT_MS} ms)")
    print(f"  cost         : ${cost_per_m:.3f} per 1M output tokens")
    print(f"  eval accuracy: {accuracy*100:.0f}%")
else:
    print("No tested level met the SLO. Lower the load, tune the server (Module 5),")
    print("or relax the SLO. The data tells you which lever you have.")

**What you should see:** a clean recommendation block with the configuration, the concurrency, and throughput, latency, cost, and accuracy together. On the workshop hardware this lands near concurrency 64 under a 500 ms p95 SLO at roughly 90% eval. That is the artifact you defend. If nothing met the SLO, the message points you back to tuning or a different model.

## Things to know

- **A benchmark number is not an SLO.** The sweep tells you what the server can do; the SLO tells you what you will allow. Always report the point under your latency line, not the prettiest number in the table.
- **p95, not the mean.** A mean latency hides the slow tail your users actually feel. Pick the point under the line at the percentile you promise, here p95 TTFT.
- **The KV cache gauge tells you why the knee is where it is.** vLLM's V1 engine renamed `vllm:gpu_cache_usage_perc` to `kv_cache_usage_perc`; `metrics.snapshot()` returns it under both names, so your reads from earlier modules still work. Watch it climb across the sweep to see whether you are bound by KV memory or by the batch cap.
- **Cost per token only beats hosted at high utilization.** A near-idle GPU is expensive per token because the hourly cost is fixed. The cheapest row is always the busiest one that still meets the SLO.
- **The eval set must look like your traffic.** Ten trivia questions prove the method, not your product. Replace them with prompts and checks from your real workload before you trust the accuracy number.
- **What healthy latency looks like.** As a rough anchor, a small model on one modern GPU serves TTFT in the tens of milliseconds at low load, rising into the hundreds near the knee, with inter-token latency of a few to low tens of milliseconds (an ITL of 10 ms is 100 tokens per second per user). TTFT in seconds at low concurrency means something is misconfigured, not saturated.
- **A wide p50-to-p99 gap means preemption, not slowness.** If the median TTFT is healthy but the p99 is several times larger, the tail is hitting the queue and preemption, not steady service. That points back to Module 5: raise `--gpu-memory-utilization` or shrink the model so the cache stops evicting. Throughput flat while the batch grows points the other way, to the roofline limit, where quantization is the lever.

> NOTE: Run the sweep against the configuration you will actually deploy. Benchmarking an untuned server and shipping a tuned one, or the reverse, makes the recommendation a fiction.

## Try it yourself

**Move the SLO and watch the operating point shift.** Set `SLO_TTFT_MS` to 250, then to 1000, and re-run the recommendation. A tighter promise costs throughput and raises your cost per token; a looser one lets you pack the card. The starter cell prints the new pick. **Stretch:** plot both SLOs on the same curve and mark where each lands.

**Price your own instance.** Replace `INSTANCE_HOURLY` with the price of the GPU you provisioned and read the cost per million tokens at your chosen point. Set it beside the hosted bill from Module 1 to find your crossover for real.

**Bring your own eval set.** Swap the `EVAL` list for ten prompts from your product, each with a substring that marks a correct answer, and re-score. Now the accuracy column is about your traffic, not trivia.

In [ ]:
# Change SLO_TTFT_MS and INSTANCE_HOURLY, then run the cell.
SLO_TTFT_MS = 250        # your latency objective in ms
INSTANCE_HOURLY = 1.50   # your GPU instance price per hour

under = [r for r in rows if r["p95_ttft_ms"] and r["p95_ttft_ms"] <= SLO_TTFT_MS]
if under:
    pick = max(under, key=lambda r: r["output_throughput"])
    cpm = (INSTANCE_HOURLY / (pick["output_throughput"] * 3600)) * 1e6
    print(f"under SLO {SLO_TTFT_MS} ms -> c={pick['concurrency']}  "
          f"{pick['output_throughput']:.0f} tok/s  ${cpm:.3f}/1M tok")
else:
    print(f"No level met SLO {SLO_TTFT_MS} ms. Tune the server or relax the SLO.")

## A note on GuideLLM

`vllm bench serve` covers everything above. When you want constant-rate load (requests per second rather than fixed concurrency), per-percentile breakdowns, and automatic rate sweeps, GuideLLM is the richer tool. Install it with `pip install guidellm` and run a sweep:

```bash
guidellm benchmark --target "$VLLM_HOST" --model "$MODEL_NAME" --rate-type sweep --max-seconds 60
```

It produces the same throughput-versus-latency curve with more detail. Reach for it when you are sizing a real deployment.

## Summary

- A benchmark number says what the server can do; an SLO says what you will allow. The sweep gives the curve, the SLO picks the point.
- The operating point is the rightmost point under your latency line: the most throughput you can serve without breaking the promise.
- Throughput plus instance price is cost per million tokens, which falls as you pack the card and is how you compare with hosted pricing on equal terms.
- A small eval set puts accuracy in the same table as speed and cost, so the decision is all three at one point.
- The deliverable is one operating point with throughput, latency, cost, and accuracy: the artifact you defend.

## Next

**Module 9: Agents on Kubernetes.** You tuned the inference and picked the point you will run at. Next you put an agent on top of it, deploy it on the GPU you own, and talk to the server you spent this workshop earning.